# Exploratory Data Analysis in Action - EDA: Energy production

Energetic production questions:
- Q1: How much energy was estimated to be produced in 2020 by nuclear power plants?
- Q2: Which country produces the most energy using nuclear power plants?
- Q3: Which country produces the most energy using solar power plants?
- Q4: Which nuclear power plant produces the most energy? How many of the largest solar power plants (sorted by energy production) are needed to replace it?
- Q5: Where are the power plants in Q4 located? Plot a map with their location!
- Q6: _Come up with your own question_

In [1]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import sys
sys.path.append("../src/")

from helper import cuteplot
from helper import minmax_scaler
plt.rcParams["figure.figsize"] = [22,10]

In [2]:
import pickle
gdf_world = pickle.load(open( "../data/gdf_world.p", "rb" ))
gdf_europe = pickle.load(open( "../data/gdf_europe.p", "rb" ))
gdf_germany = pickle.load(open( "../data/gdf_germany.p", "rb" ))

> **Q1: How much energy was estimated to be produced in 2020 by nuclear power plants?**

In [4]:
gdf_world.columns

Index(['country code', 'country', 'name of powerplant', 'capacity in MW',
       'latitude', 'longitude', 'primary_fuel', 'start date', 'owner of plant',
       'geolocation_source', 'estimated_generation_gwh_2020', 'green',
       'CONTINENT', 'geometry'],
      dtype='object')

In [6]:
gdf_world['primary_fuel'].unique()

array(['Hydro', 'Solar', 'Gas', 'Other', 'Oil', 'Nuclear', 'Coal', 'Wind',
       'Waste', 'Biomass', 'Wave and Tidal', 'Geothermal', 'Storage',
       'Cogeneration', 'Petcoke'], dtype=object)

In [9]:
gdf_world.loc[gdf_world['primary_fuel']=='Nuclear']['estimated_generation_gwh_2020'].sum()

np.float64(2325381.15)

In [10]:
gdf_world.loc[gdf_world['primary_fuel']=='Coal']['estimated_generation_gwh_2020'].sum()

np.float64(8814356.030000001)

> **Q2: Which country produces the most energy using nuclear power plants?**

In [16]:
gdf_world.loc[gdf_world['primary_fuel']=='Nuclear'].groupby('country')['estimated_generation_gwh_2020'].sum().sort_values(ascending=False)

country
United States of America    760774.22
France                      365634.96
China                       202297.87
Russia                      201157.66
Japan                       156060.35
Canada                       94670.18
Ukraine                      85298.53
Germany                      75073.97
Sweden                       50927.73
Spain                        48531.59
Belgium                      41142.50
United Kingdom               36245.09
India                        33059.17
Czech Republic               29100.09
Switzerland                  23050.67
Bulgaria                     15544.99
Brazil                       15189.42
Hungary                      15111.29
Finland                      13929.40
Slovakia                     13611.56
South Africa                 13236.99
Romania                      10374.08
Iran                          7514.00
Argentina                     6212.98
Slovenia                      6017.00
Netherlands                   3203.82
Arme

> **Q3: Which country produces the most energy using solar power plants?**

In [17]:
gdf_world.loc[gdf_world['primary_fuel']=='Solar'].groupby('country')['estimated_generation_gwh_2020'].sum().sort_values(ascending=False)

country
China                       88088.32
United States of America    73433.82
India                       53537.85
United Kingdom              12878.64
Spain                       10154.37
                              ...   
Palestine                      15.01
Niger                          12.95
French Guiana                   9.84
Suriname                        9.81
Costa Rica                      9.65
Name: estimated_generation_gwh_2020, Length: 88, dtype: float64

> **Q4: Which nuclear power plant produces the most energy? How many of the largest solar power plants (sorted by energy production) are needed to replace it?**

> Visualize the estimated generation for each of this powerplants in one bar plot.

In [21]:
gdf_world.loc[gdf_world['primary_fuel']=='Nuclear'].groupby('country')['estimated_generation_gwh_2020'].count()

country
Argentina                    3
Armenia                      1
Belgium                      3
Brazil                       2
Bulgaria                     1
Canada                       5
China                       10
Czech Republic               2
Finland                      1
France                      18
Germany                      7
Hungary                      1
India                        6
Iran                         1
Japan                        8
Netherlands                  1
Pakistan                     0
Romania                      1
Russia                      10
Slovakia                     2
Slovenia                     1
South Africa                 1
Spain                        4
Sweden                       2
Switzerland                  4
Ukraine                      4
United Kingdom               5
United States of America    54
Name: estimated_generation_gwh_2020, dtype: int64

In [35]:
gdf_world['country'].unique()

array(['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina',
       'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bangladesh',
       'Belarus', 'Belgium', 'Benin', 'Bhutan', 'Bolivia',
       'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi',
       'Cambodia', 'Cameroon', 'Canada', 'Central African Republic',
       'Chile', 'China', 'Colombia', 'Congo', 'Costa Rica',
       'Cote DIvoire', 'Croatia', 'Cuba', 'Cyprus', 'Czech Republic',
       'Democratic Republic of the Congo', 'Denmark',
       'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador',
       'Equatorial Guinea', 'Eritrea', 'Estonia', 'Ethiopia', 'Fiji',
       'Finland', 'France', 'French Guiana', 'Gabon', 'Gambia', 'Georgia',
       'Germany', 'Ghana', 'Greece', 'Guatemala', 'Guinea',
       'Guinea-Bissau', 'Guyana', 'Honduras', 'Hungary', 'Iceland',
       'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy',
       'Jamaica',

In [36]:
gdf_country = gdf_world.loc[gdf_world['country']=='China']

In [37]:
gdf_country_nuclear = gdf_country.loc[gdf_country['primary_fuel']=='Nuclear']

In [42]:
gdf_country_nuclear.columns

Index(['country code', 'country', 'name of powerplant', 'capacity in MW',
       'latitude', 'longitude', 'primary_fuel', 'start date', 'owner of plant',
       'geolocation_source', 'estimated_generation_gwh_2020', 'green',
       'CONTINENT', 'geometry'],
      dtype='object')

In [46]:
gdf_country_nuclear[['name of powerplant','estimated_generation_gwh_2020']].sort_values(by='estimated_generation_gwh_2020', ascending=False)[:1]

,name of powerplant,estimated_generation_gwh_2020
6193,Hongyanhe,30881.4


In [28]:
for country in gdf_world['country'].unique():
    gdf_country = gdf_world.loc[gdf_world['country']==country]
    gdf_country_nuclear = gdf_country.loc[gdf_country['primary_fuel']=='Nuclear']
    gdf_country_nuclear.sort_values(by=gdf_country_nuclear['estimated_generation_gwh_2020'], ascending=False)

KeyError: Series([], Name: estimated_generation_gwh_2020, dtype: float64)

> **Q5: Where are the power plants in Q4 located? Plot a map with their location!** 

> **Q6: _Come up with your own question_**